# Entrenamiento del reconocedor de señas

Entra un `.npz` con secuencias de 30 frames × 128 valores y sale un modelo
Keras más sus métricas.

Se entrenan **dos** modelos:
1. **base**, solo con LSC-54, como referencia;
2. **unificado**, con todas las fuentes.

Sin la referencia no hay forma de saber si sumar LSC50 ayudó o estorbó.

**Antes de correr:** Entorno de ejecución → Cambiar tipo → GPU T4.

In [ ]:
import os

REPO = 'https://github.com/mauro-1508/backendTS.git'
RAMA = 'feat/backend-dominios-ia'

if not os.path.exists('train_model.py'):
    !git clone -q --branch {RAMA} --depth 1 {REPO} repo
    !cp repo/src/domains/ia/training/extraction/*.py .
    !cp repo/src/domains/ia/training/extraction/labels.json .

import tensorflow as tf
print('tensorflow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'no hay (irá lento)')

## 1. Datos

Se toman los `.jsonl` que dejaron los notebooks de extracción en Drive y se
arma el `.npz`: glosas unificadas, división **por señante** y aumento solo
sobre el conjunto de entrenamiento.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATOS = '/content/drive/MyDrive/traduce_senas/datasets'
!mkdir -p datasets
!cp {DATOS}/lsc54.jsonl datasets/ 2>/dev/null || echo 'falta lsc54.jsonl'
!cp {DATOS}/lsc50.jsonl datasets/ 2>/dev/null || echo 'falta lsc50.jsonl'
!ls -lh datasets/

In [ ]:
# build_dataset.py busca las fuentes en rutas relativas; se ajustan aquí.
import build_dataset
build_dataset.DEFAULT_SOURCES = [
    ('lsc54', 'datasets/lsc54.jsonl'),
    ('lsc50', 'datasets/lsc50.jsonl'),
]

!python -c "import build_dataset; build_dataset.DEFAULT_SOURCES=[('lsc54','datasets/lsc54.jsonl'),('lsc50','datasets/lsc50.jsonl')]; build_dataset.main()" --plantilla-labels
print('\nRevisa labels.json: las clases sin glosa quedan fuera del entrenamiento.')

In [ ]:
import json

labels = json.load(open('labels.json', encoding='utf-8'))
for ds, mapa in labels.items():
    sin = [k for k, v in mapa.items() if not v]
    print(f'{ds}: {len(mapa)} clases, {len(sin)} sin glosa')
    if sin:
        print('   ', sin[:10], '...' if len(sin) > 10 else '')

In [ ]:
import sys
sys.argv = ['build_dataset.py', '--out', 'datasets/dataset.npz']
build_dataset.main()

## 2. Modelo base: solo LSC-54

La referencia contra la que se compara todo lo demás.

In [ ]:
import train_model as tm

datos_base = tm.load_dataset('datasets/dataset.npz', solo_dataset='lsc54')
print('glosas:', len(datos_base['glosses']))
for c in ('train', 'val', 'test'):
    X, y, s = datos_base[c]
    print(f'  {c}: {len(X)} secuencias, {len(set(s.tolist()))} señantes')

modelo_base, hist_base = tm.entrenar(datos_base, epocas=60)
res_base = {
    'val': tm.report(modelo_base, *datos_base['val'][:2], datos_base['glosses'], 'BASE · validación'),
    'test': tm.report(modelo_base, *datos_base['test'][:2], datos_base['glosses'], 'BASE · prueba'),
}

## 3. Modelo unificado

In [ ]:
datos = tm.load_dataset('datasets/dataset.npz')
print('glosas:', len(datos['glosses']))

modelo, hist = tm.entrenar(datos, epocas=80)
res = {
    'val': tm.report(modelo, *datos['val'][:2], datos['glosses'], 'UNIFICADO · validación'),
    'test': tm.report(modelo, *datos['test'][:2], datos['glosses'], 'UNIFICADO · prueba'),
}

## 4. Curvas y matriz de confusión

Si la exactitud de entrenamiento se despega de la de validación, está
memorizando señantes en vez de aprender señas.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, (nombre, h) in zip(ejes, [('base (LSC-54)', hist_base), ('unificado', hist)]):
    eje.plot(h['accuracy'], label='entrenamiento')
    if 'val_accuracy' in h:
        eje.plot(h['val_accuracy'], label='validación')
    eje.set_title(nombre); eje.set_xlabel('época'); eje.set_ylabel('exactitud')
    eje.legend(); eje.grid(alpha=.3)
plt.tight_layout(); plt.show()

X_test, y_test, _ = datos['test']
if len(X_test):
    pred = modelo.predict(X_test, verbose=0).argmax(axis=1)
    m = tm.confusion(y_test, pred, len(datos['glosses']))
    plt.figure(figsize=(9, 8))
    plt.imshow(m / np.maximum(m.sum(axis=1, keepdims=True), 1), cmap='viridis')
    plt.xlabel('predicho'); plt.ylabel('real'); plt.colorbar(label='proporción')
    plt.title('Matriz de confusión · prueba')
    plt.show()

## 5. ¿Aprendió señas o aprendió datasets?

Riesgo medido al comparar las fuentes: en LSC50 las manos quedan ~0,6 anchos
de hombro más abajo que en LSC-54. Si el modelo se apoya en eso, acierta en la
prueba y falla frente a la cámara. Se revisa mirando la exactitud por fuente.

In [ ]:
X_test, y_test, s_test = datos['test']
if len(X_test):
    pred = modelo.predict(X_test, verbose=0).argmax(axis=1)
    for fuente in sorted({str(v).split(':')[0] for v in s_test}):
        mask = np.array([str(v).startswith(fuente + ':') for v in s_test])
        if mask.sum():
            print(f'{fuente}: {(pred[mask] == y_test[mask]).mean():.1%} sobre {int(mask.sum())} secuencias')

## 6. Guardar

In [ ]:
SALIDA = '/content/drive/MyDrive/traduce_senas/modelos'
os.makedirs(SALIDA, exist_ok=True)

modelo.save(f'{SALIDA}/modelo_unificado.keras')
modelo_base.save(f'{SALIDA}/modelo_base_lsc54.keras')

with open(f'{SALIDA}/glosas.json', 'w', encoding='utf-8') as fh:
    json.dump(datos['glosses'], fh, ensure_ascii=False, indent=2)
with open(f'{SALIDA}/metricas.json', 'w', encoding='utf-8') as fh:
    json.dump({'base': res_base, 'unificado': res}, fh, ensure_ascii=False, indent=2)

!ls -lh {SALIDA}
print('\nLa exportación a .tflite y TensorFlow.js va en el notebook del 1 de octubre.')